# ProboSed 01 — JCORES VCD Mining and MTD Catalog
**IODP Expedition 405 — Site C0019J, Japan Trench**

Extracts stability scores from the JCORES VCD PDF and produces the MTD boundary catalog used by all downstream notebooks.

### What this notebook does:
1. Builds a depth backbone from the core summary Excel file
2. Extracts stability scores from the JCORES VCD PDF (`CORC0019.pdf`) using standardized IODP disturbance terminology and a lowest-score-wins rule
3. Identifies MTD boundaries as contiguous intervals where stability score <= 1
4. Saves a stability log CSV and MTD catalog CSV for use in downstream notebooks

### CORC0019.pdf page format:
Each page header reads:
```
Hole C0019J Core 7K, interval 139 to 142.805 m (core depth below seafloor)
```
The miner extracts core ID from `Core 7K` and depth from `interval 139 to ...`.

### Output files:
- `C0019J_VCD_stability_log.csv` — stability score per depth interval
- `C0019J_MTD_catalog.csv` — MTD boundaries for downstream notebooks
- `C0019J_stability_profile.png` — visual QC of the extracted profile

In [ ]:
# Cell 1 — Install dependencies
!pip install pymupdf openpyxl pandas numpy matplotlib --quiet

In [ ]:
# Cell 2 — Mount Google Drive and clone ProboSed
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os, shutil

# Always clone fresh so the latest labeler.py is used
repo_path = '/content/ProboSed'
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)

result = subprocess.run(
    ['git', 'clone', 'https://github.com/rocknrene/ProboSed.git', repo_path],
    capture_output=True, text=True
)
print('Clone return code:', result.returncode)
if result.returncode != 0:
    print('STDERR:', result.stderr)

sys.path.insert(0, repo_path)

# Verify the labeler is importable and lexicon is correct
from core_ml.labeler import JCORESMiner, JCORES_LEXICON, DRILLING_CAUSE_TERMS
print('\nLexicon checks:')
print(f'  minor in lexicon: {"minor" in JCORES_LEXICON}  (should be False)')
print(f'  disturb in lexicon: {"disturb" in JCORES_LEXICON}  (should be False)')
print(f'  sheared in lexicon: {"sheared" in JCORES_LEXICON}  (should be True)')
print(f'  high disturbance in lexicon: {"high disturbance" in JCORES_LEXICON}  (should be True)')
print(f'  due to drilling in DRILLING_CAUSE_TERMS: {"due to drilling" in DRILLING_CAUSE_TERMS}  (should be True)')
print(f'\nCORE_PATTERN: {JCORESMiner.CORE_PATTERN.pattern}')
print(f'DEPTH_PATTERN: {JCORESMiner.DEPTH_PATTERN.pattern}')

In [ ]:
# Cell 3 — Configure paths
import os

DATA_DIR   = '/content/drive/MyDrive/iodp/X405/Data & Data Tracking'
OUTPUT_DIR = '/content/drive/MyDrive/iodp/X405/ProboSed_Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# CORC0019.pdf is the JCORES VCD for Site C0019J
VCD_PDF      = os.path.join(DATA_DIR, 'vcds', 'CORC0019.pdf')
SUMMARY_XLSX = os.path.join(DATA_DIR, '405-C0019J_CoreSummary_2410280924.xlsx')

STABILITY_CSV = os.path.join(OUTPUT_DIR, 'C0019J_VCD_stability_log.csv')
MTD_CSV       = os.path.join(OUTPUT_DIR, 'C0019J_MTD_catalog.csv')
PROFILE_PNG   = os.path.join(OUTPUT_DIR, 'C0019J_stability_profile.png')

for path, label in [(VCD_PDF, 'VCD PDF'), (SUMMARY_XLSX, 'Core summary Excel')]:
    status = 'FOUND' if os.path.exists(path) else 'NOT FOUND — check path'
    print(f'{label}: {status}')
    print(f'  {path}')

In [ ]:
# Cell 4 — Verify PDF text format
# Reads the first 3 pages to confirm the miner can find core IDs and depths.
# Expected output: Core=7K (or similar), Depth=139.0 (or similar)
import fitz, re
from core_ml.labeler import JCORESMiner

doc = fitz.open(VCD_PDF)
print(f'PDF pages: {len(doc)}')
print()

for i in range(min(3, len(doc))):
    page = doc.load_page(i)
    text = page.get_text()
    lines = [l.strip() for l in text.split('\n') if l.strip()][:5]
    print(f'Page {i+1} header:')
    for l in lines:
        print(f'  {l}')

    core_m  = JCORESMiner.CORE_PATTERN.search(text)
    depth_m = JCORESMiner.DEPTH_PATTERN.search(text)
    core_id = (core_m.group(1) or core_m.group(2)).upper() if core_m else None
    depth   = float(depth_m.group(1)) if depth_m else None
    print(f'  -> Core={core_id}  Depth={depth}')
    print()

doc.close()

if core_id is None:
    print('WARNING: Core ID not found on first pages.')
    print('Check that VCD_PDF path is correct and the PDF is CORC0019.pdf.')
else:
    print('Core ID and depth extraction working correctly.')

In [ ]:
# Cell 5 — Build depth backbone from core summary
from core_ml.labeler import JCORESMiner

miner    = JCORESMiner(VCD_PDF, SUMMARY_XLSX)
backbone = miner.build_backbone()

print(f'\nBackbone sample (first 10 rows):')
print(backbone.head(10).to_string(index=False))

In [ ]:
# Cell 6 — Extract stability scores from the VCD PDF
# This uses the updated miner with:
#   - CORE_PATTERN matching 'Core 7K' format
#   - DEPTH_PATTERN matching 'interval 139 to 142.805 m' format
#   - Minimum-score rule (most severe disturbance wins)
#   - Drilling cause detection via DRILLING_CAUSE_TERMS
vcd_df = miner.extract(backbone)

print(f'\nStability score distribution:')
print(vcd_df['Stability'].value_counts().sort_index())
print(f'\nDepth-matched pages: {vcd_df["Depth_m"].notna().sum()} / {len(vcd_df)}')
print(f'Drilling artifacts flagged: {vcd_df["Drilling_Artifact"].sum()}')
print(f'\nFirst 10 rows:')
print(vcd_df[['PDF_Page','Core','Depth_m','Stability','Disturbance','Drilling_Artifact']].head(10).to_string(index=False))

In [ ]:
# Cell 7 — Spot check: verify the four previously problematic cores
print('Spot-check of previously problematic cores:')
print('(Expected: 14K~scaly/1, 50K~bioturbated/3, 60K~highly disturbed/0, 87K~sheared/1)\n')

for core in ['14K', '50K', '60K', '87K']:
    rows = vcd_df[vcd_df['Core'] == core]
    if not rows.empty:
        r = rows.iloc[0]
        print(f'  {core} ({r["Depth_m"]:.1f} mbsf): '
              f'score={r["Stability"]}  term="{r["Disturbance"]}"  '
              f'drilling={r["Drilling_Artifact"]}')
    else:
        print(f'  {core}: not found in output')

# also check unmatched pages
unmatched = vcd_df[vcd_df['Depth_m'].isna()]
print(f'\nUnmatched pages (no depth assigned): {len(unmatched)} / {len(vcd_df)}')
if len(unmatched) > 0 and len(unmatched) < 10:
    print(unmatched[['PDF_Page','Core','Disturbance','Snippet']].to_string(index=False))

In [ ]:
# Cell 8 — Save stability log
vcd_df.to_csv(STABILITY_CSV, index=False)
print(f'Saved: {STABILITY_CSV}')
print(f'Rows: {len(vcd_df)}')

In [ ]:
# Cell 9 — Build MTD boundary catalog
# MTD intervals are contiguous depth sequences where stability <= 1
# (scaly fabric = 1 or slurried/chaotic = 0)
# Drilling artifact intervals are excluded before catalog construction.

# exclude drilling artifacts from catalog
geo_df = vcd_df[~vcd_df['Drilling_Artifact']].copy()
mtd_catalog = JCORESMiner.score_to_mtd_catalog(geo_df, stability_threshold=1)

mtd_catalog.to_csv(MTD_CSV, index=False)
print(f'MTD intervals identified: {len(mtd_catalog)}')
print(f'Saved: {MTD_CSV}')
print()
if len(mtd_catalog) > 0:
    print(mtd_catalog.to_string(index=False))

In [ ]:
# Cell 10 — Plot stability profile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plot_df = vcd_df[~vcd_df['Drilling_Artifact']].dropna(subset=['Depth_m','Stability']).sort_values('Depth_m')

fig, ax = plt.subplots(figsize=(6, 16))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f7f7f7')

ax.step(plot_df['Stability'], plot_df['Depth_m'],
        where='post', color='#2c3e50', lw=1.5, zorder=3)

# shade MTD intervals
for _, row in mtd_catalog.iterrows():
    ax.axhspan(row['top_m'], row['bottom_m'],
               alpha=0.2, color='#cc2222', zorder=1)

ax.axhline(820, color='#ff4444', ls='--', lw=1.5, alpha=0.8,
           label='Plate boundary fault (~820 mbsf)')

ax.invert_yaxis()
ax.set_xlim(-0.5, 3.5)
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['Slurried\n(0)', 'Scaly\n(1)', 'Coherent\n(2)', 'Intact\n(3)'], fontsize=10)
ax.set_ylabel('Depth (m CSF-A)', fontsize=12)
ax.set_title('C0019J Stability Profile\n(JCORESMiner extraction)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
ax.text(0.02, 0.02, f'MTD intervals: {len(mtd_catalog)}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#cc2222', alpha=0.2))

plt.tight_layout()
plt.savefig(PROFILE_PNG, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {PROFILE_PNG}')

In [ ]:
# Cell 11 — Summary
print('=== MTD CATALOG SUMMARY ===')
print(f'Total MTD intervals:     {len(mtd_catalog)}')
if len(mtd_catalog) > 0:
    print(f'Shallowest top:          {mtd_catalog["top_m"].min():.1f} m CSF-A')
    print(f'Deepest base:            {mtd_catalog["bottom_m"].max():.1f} m CSF-A')
    print(f'Mean thickness:          {mtd_catalog["thickness_m"].mean():.1f} m')
    print(f'Total MTD thickness:     {mtd_catalog["thickness_m"].sum():.1f} m')
    print()
    print('Individual intervals:')
    for _, row in mtd_catalog.iterrows():
        print(f'  {row["mtd_id"]}: {row["top_m"]:.1f} - {row["bottom_m"]:.1f} m  '
              f'({row["thickness_m"]:.1f} m thick, '
              f'mean stability = {row["mean_stability"]:.1f})')
print()
print(f'Drilling artifact intervals excluded: {vcd_df["Drilling_Artifact"].sum()}')
print(f'Depth-unmatched pages: {vcd_df["Depth_m"].isna().sum()}')
print()
print('Output files:')
print(f'  {STABILITY_CSV}')
print(f'  {MTD_CSV}')
print(f'  {PROFILE_PNG}')